In [ ]:
# Import standard libraries
import os
import re

# Import third-party libraries
import joblib
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer


# ---------------------------
# Configuration
# ---------------------------
def get_config():
    # Define generation parameters
    gen_params = {
        "num_beams": 5,
        "max_new_tokens": 512,
        "length_penalty": 1.096,
        "early_stopping": True,
        "do_sample":False, 
        "min_length":20,            
        "repetition_penalty":1.0
    }

    # Pick device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Return full config
    return {
        "data_path": "/kaggle/input/deep-past-initiative-machine-translation/test.csv",
        "models": [
            "/kaggle/input/byt5-akkadian-en-v5/pytorch/default/1/byt5-akkadian-en-v5",
            "/kaggle/input/byt5-akkadian-en-v6/pytorch/default/1/byt5-akkadian-en-V6",
            "/kaggle/input/byt5-akkadian-en-v7/pytorch/default/1/byt5-akkadian-en-V7",
        ],
        "model_weights": [0.99, 0.90, 0.99],
        "device": device,
        "max_len": 512,
        "batch_size": 8,
        "gen_params": gen_params,
    }


# ---------------------------
# Preprocess and postprocess
# ---------------------------
def preprocess_transliteration(text):
    # Return empty string for missing values
    if pd.isna(text):
        return ""

    # Convert to string
    processed_text = str(text)

    # Normalize large gaps
    processed_text = re.sub(r"(\.{3,}|…+|……)", "<gap>", processed_text)

    # Normalize small gaps
    processed_text = re.sub(r"(xx+|\s+x\s+)", "<gap>", processed_text)

    return processed_text


def postprocess_translation(text):
    # Return empty string for invalid outputs
    if not isinstance(text, str) or not text.strip():
        return ""

    # Normalize specific diacritics
    processed_text = text.replace("ḫ", "h").replace("Ḫ", "H")

    # Convert subscript digits to normal digits
    sub_map = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789")
    processed_text = processed_text.translate(sub_map)

    # Normalize gap markers
    processed_text = re.sub(r"(\[x\]|\(x\)|\bx\b)", "<gap>", processed_text, flags=re.I)
    processed_text = re.sub(r"(\.{3,}|…|\[\.+\])", "<gap>", processed_text)

    # Merge adjacent gaps
    processed_text = re.sub(r"<gap>\s*<gap>", " <gap> ", processed_text)
    #processed_text = re.sub(r"<big_gap>\s*<big_gap>", " <big_gap> ", processed_text)

    # Remove some parenthetical morphology notes
    processed_text = re.sub(
        r"\((fem|plur|pl|sing|singular|plural|\?|!)\.?\s*\w*\)",
        "",
        processed_text,
        flags=re.I,
    )
    

    # collapse duplicate gaps
    processed_text = re.sub(r"(<gap>\s*)+", "<gap> ", text)

    # remove unwanted morphology tags
    

    # Temporarily protect tokens from cleanup
    processed_text = processed_text.replace("<gap>", "\x00GAP\x00")
    

    # Remove bad characters
    bad_chars = '!?()"—–<>⌈⌋⌊[]+ʾ/;'
    processed_text = processed_text.translate(str.maketrans("", "", bad_chars))

    # Restore tokens
    processed_text = processed_text.replace("\x00GAP\x00", " <gap> ")
    

    # Handle common fractions from decimals
    frac_map = {
        r"\.5\b": " ½",
        r"\.25\b": " ¼",
        r"\.75\b": " ¾",
        r"\.33+\d*\b": " ⅓",
        r"\.66+\d*\b": " ⅔",
    }
    for pattern, replacement in frac_map.items():
        processed_text = re.sub(r"(\d+)" + pattern, r"\1" + replacement, processed_text)
        processed_text = re.sub(r"\b0" + pattern, replacement.strip(), processed_text)

    # Remove repeated single words
    processed_text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", processed_text)

    # Remove repeated n-grams
    for n in range(4, 1, -1):
        pattern = r"\b((?:\w+\s+){" + str(n - 1) + r"}\w+)(?:\s+\1\b)+"
        processed_text = re.sub(pattern, r"\1", processed_text)

    # Normalize whitespace and trim
    processed_text = re.sub(r"\s+", " ", processed_text).strip().strip("-")

    return processed_text


# ---------------------------
# Model soup
# ---------------------------
def normalize_model_weights(model_weights):
    # Compute sum of weights
    total_score = sum(model_weights)

    # Return normalized weights
    return [w / total_score for w in model_weights]


def load_state_dict(model_path):
    # Load model and return its state dict
    model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
    return model.state_dict()


def create_model_soup(model_paths, model_weights, device):
    # Normalize weights
    normalized_weights = normalize_model_weights(model_weights)

    # Load template model
    template_model = AutoModelForSeq2SeqLM.from_pretrained(model_paths[1])
    soup_state_dict = template_model.state_dict()

    # Load other state dicts
    model_1_state_dict = load_state_dict(model_paths[0])
    model_3_state_dict = load_state_dict(model_paths[2])

    # Build soup for each key
    for key in soup_state_dict:
        # Initialize with weighted template value
        weighted_value = normalized_weights[1] * soup_state_dict[key]
        norm_factor = normalized_weights[1]

        # Add model 1 contribution if available
        if key in model_1_state_dict:
            weighted_value += normalized_weights[0] * model_1_state_dict[key]
            norm_factor += normalized_weights[0]

        # Add model 3 contribution if available
        if key in model_3_state_dict:
            weighted_value += normalized_weights[2] * model_3_state_dict[key]
            norm_factor += normalized_weights[2]

        # Assign normalized combined tensor
        soup_state_dict[key] = weighted_value / norm_factor

    # Load soup weights
    template_model.load_state_dict(soup_state_dict)

    # Move to device and set eval
    template_model = template_model.to(device)
    template_model = template_model.eval()
    template_model = template_model.float()

    return template_model


# ---------------------------
# Dataset and dataloader
# ---------------------------
class AkkadianTranslationDataset(Dataset):
    # Initialize dataset
    def __init__(self, dataframe):
        # Store ids
        self.ids = dataframe["id"].tolist()

        # Build prompted texts
        self.texts = [
            "Translate Akkadian to English: " + str(t)
            for t in dataframe["transliteration"]
        ]

    # Return dataset length
    def __len__(self):
        return len(self.ids)

    # Return one item
    def __getitem__(self, idx):
        return self.ids[idx], self.texts[idx]


def collate_batch(batch, tokenizer, max_len):
    # Extract ids
    ids = [item[0] for item in batch]

    # Extract texts
    texts = [item[1] for item in batch]

    # Tokenize
    tokenized = tokenizer(
        texts,
        max_length=max_len,
        padding=True,
        truncation=True,
        return_tensors="pt",
    )

    return ids, tokenized


def build_dataloader(dataframe, tokenizer, batch_size, max_len):
    # Create dataset
    dataset = AkkadianTranslationDataset(dataframe)

    # Create dataloader
    data_loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        collate_fn=lambda batch: collate_batch(batch, tokenizer, max_len),
    )

    return data_loader


# ---------------------------
# Inference
# ---------------------------
def run_inference(model, tokenizer, data_loader, device, gen_params):
    # Initialize outputs
    inference_results = []

    # Disable gradients
    with torch.inference_mode():
        # Iterate batches
        for ids, inputs in data_loader:
            # Move inputs to device
            input_ids = inputs.input_ids.to(device)
            attention_mask = inputs.attention_mask.to(device)

            # Generate
            outputs = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                **gen_params,
            )

            # Decode
            decoded_texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

            # Postprocess
            cleaned_translations = [
                postprocess_translation(text) for text in decoded_texts
            ]

            # Store results
            inference_results.extend(zip(ids, cleaned_translations))

    return inference_results


# ---------------------------
# I/O helpers
# ---------------------------
def load_data(data_path):
    # Read csv
    dataframe = pd.read_csv(data_path)

    # Preprocess transliteration
    dataframe["transliteration"] = dataframe["transliteration"].apply(
        preprocess_transliteration
    )

    return dataframe


def save_submission(inference_results, output_path):
    # Build submission dataframe
    submission_df = pd.DataFrame(inference_results, columns=["id", "translation"])

    # Save
    submission_df.to_csv(output_path, index=False)

    return submission_df


# ---------------------------
# Main
# ---------------------------
def main():
    # Load config
    config = get_config()

    # Load data
    dataframe = load_data(config["data_path"])

    # Create model soup
    model = create_model_soup(
        model_paths=config["models"],
        model_weights=config["model_weights"],
        device=config["device"],
    )
    

    # Load tokenizer from template model
    tokenizer = AutoTokenizer.from_pretrained(config["models"][1])

    # Build dataloader
    data_loader = build_dataloader(
        dataframe=dataframe,
        tokenizer=tokenizer,
        batch_size=config["batch_size"],
        max_len=config["max_len"],
    )

    # Run inference
    inference_results = run_inference(
        model=model,
        tokenizer=tokenizer,
        data_loader=data_loader,
        device=config["device"],
        gen_params=config["gen_params"],
    )

    # Save submission
    submission_df = save_submission(inference_results, "/kaggle/working/submission.csv")

    # Print sample
    print(submission_df.head(10).to_string(index=False))


if __name__ == "__main__":
    main()                                                                            